# Teste isolado — Fitch Ratings (Brasil, RACs + Research)

Fonte candidata: **Fitch Ratings**, agência de rating. Setor
Regulatório/Múltiplo (cobre empresas de vários setores de infra — energia,
saneamento, transporte, telecom). Notebook **descartável** (Fase 1) --
sem dispatcher, sem `atualizar_status_fonte`, sem gravar nada em
`controle_fontes` (registro fica pendente de decisão sobre integração,
por instrução explícita).

Dois URLs, mesma fonte:
1. `?expanded=racs&filter.language=Portuguese&filter.country=Brazil` --
   Rating Action Commentary (ações de rating)
2. `?expanded=research&filter.language=Portuguese&filter.country=Brazil`
   -- Insights (Rating Reports, Special Reports, etc.)

## Passo 0 — checagem de bloqueio anti-bot (feita ANTES de qualquer outra coisa)

Instrução explícita do pedido: abrir só os dois links primeiro e verificar
sinais de bloqueio (CAPTCHA, "access denied", página de verificação)
antes de prosseguir -- sem tentar nenhuma técnica de contorno se algo
aparecesse.

**Resultado: nenhum sinal de bloqueio em nenhuma das duas URLs.**
Testado com Selenium + Chrome headless real (não `curl_cffi`/impersonation
de TLS como em outras fontes -- aqui o pedido foi explicitamente por
navegador real, usando o cluster pessoal com o init script do Selenium já
resolvido: `chrisaraujofsz@gmail.com's Selenium Personal Cluster`,
`/Volumes/desafio_kinea/utils/selenium/init_selenium2_desafio.sh`).
Nenhuma das strings de bloqueio checadas (captcha, access denied,
cloudflare challenge, tráfego incomum, 403, etc.) apareceu no HTML nem no
texto visível de nenhuma das páginas. Ambas carregaram conteúdo real:
"1,116 Results" na página de RACs, "1,948 Results" na de Research --
títulos, datas e resumos de rating actions/relatórios reais e atuais.

**Nota técnica** (não é bloqueio, é config do cluster): o cluster pessoal
de Selenium não aceita workload de Job (`does not support jobs
workload`) -- só execução interativa. A investigação abaixo foi feita via
Command Execution API (`/api/1.2/contexts` + `/api/1.2/commands`) em vez
de `databricks jobs submit`, mas o notebook abaixo roda normalmente
anexado ao cluster pelo Workspace UI, como qualquer outro.

## Confirmado — estrutura da listagem

Ambas as páginas usam o mesmo template (`div.frw-column` por item):

- **Tipo + data**: `.frw-heading--tag` -- texto combinado, ex. `"Rating
  Action Commentary / Tue 18 Aug, 2026"` (RACs) ou `"Rating Report / Tue
  18 Aug, 2026"` (Research) -- precisa separar tipo (antes do `/`) de
  data (depois do `/`, formato `"Www DD Mon, YYYY"`, em inglês mesmo com
  `filter.language=Portuguese`).
- **Título + link**: `h3.frw-heading--5 > a` -- `href` relativo
  (ex. `/research/pt/banks/banco-citibank-sa-18-08-2026`), precisa
  `urljoin` com `https://www.fitchratings.com`.
- **Resumo**: `p` (irmão do `h3`, mesmo `div.frw-column`) -- texto
  truncado com "...", não é o texto completo.
- **Nome da empresa avaliada**: não é um campo estruturado separado --
  nos itens de **RAC**, geralmente aparece embutido no título (ex.
  "...do Assaí...", "...da Eneva..."); nos itens de **Research** do tipo
  "Rating Report", o título **é** o nome da empresa (ex. "Banco Citibank
  S.A."); em outros tipos de Research (Special Report, Non-Rating Action
  Commentary), o título é temático, não nominal. Extrair "empresa
  avaliada" de forma confiável exigiria parsing mais elaborado (ex. NER,
  ou abrir a página de detalhe e ler a seção `ENTITIES`/`RATING ACTIONS`
  -- ver Teste 2) -- não dá pra assumir que o título sozinho sempre
  contém isso de forma extraível por regex simples.

24 itens por página em ambas as URLs testadas (paginação não investigada
nesta Fase 1 -- não fazia parte do pedido).

In [0]:
%pip install --quiet beautifulsoup4 lxml
# Neste cluster (pessoal, com init script do Selenium), Selenium e Chrome
# já vêm prontos via init script -- não precisa instalar/reiniciar
# Python para eles.

In [0]:
import re
import time
import urllib.parse
from typing import Optional

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

In [0]:
# =============================================================================
# Configuração
# =============================================================================

BASE_URL = "https://www.fitchratings.com"
URLS = {
    "racs": f"{BASE_URL}/search/?expanded=racs&filter.language=Portuguese&filter.country=Brazil",
    "research": f"{BASE_URL}/search/?expanded=research&filter.language=Portuguese&filter.country=Brazil",
}

CHROME_BIN = "/tmp/chrome/chrome-linux/chrome"
CHROMEDRIVER_BIN = "/tmp/chrome/chromedriver_linux64/chromedriver"

SINAIS_BLOQUEIO = [
    "captcha", "recaptcha", "hcaptcha",
    "access denied", "acesso negado",
    "are you a human", "verify you are human",
    "unusual traffic", "tráfego incomum",
    "cloudflare", "checking your browser", "just a moment",
    "px-captcha", "perimeterx", "datadome",
    "forbidden", "403 forbidden",
]

MESES_EN = {
    "jan": "01", "feb": "02", "mar": "03", "apr": "04", "may": "05", "jun": "06",
    "jul": "07", "aug": "08", "sep": "09", "oct": "10", "nov": "11", "dec": "12",
}
PADRAO_DATA_FITCH = re.compile(r"(\d{1,2}) (\w{3}), (\d{4})")

In [0]:
def novo_driver():
    options = Options()
    options.binary_location = CHROME_BIN
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--window-size=1366,900")
    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    )
    service = Service(executable_path=CHROMEDRIVER_BIN)
    return webdriver.Chrome(service=service, options=options)

## Passo 0 (código) — checagem de bloqueio

Reproduz a checagem já feita: abre as duas URLs, procura sinais de
bloqueio conhecidos no HTML/texto visível. Se qualquer sinal aparecer,
o código para (`assert`) em vez de seguir adiante -- mesma regra do
pedido original, mantida aqui de forma executável.

In [0]:
driver = novo_driver()

paginas_html = {}

for nome, url in URLS.items():
    print(f"\n=== {nome}: {url} ===")
    driver.get(url)
    time.sleep(6)

    html = driver.page_source
    texto_visivel = driver.find_element(By.TAG_NAME, "body").text
    paginas_html[nome] = html

    encontrados = [s for s in SINAIS_BLOQUEIO if s in html.lower() or s in texto_visivel.lower()]
    print(f"  título: {driver.title!r} | chars HTML: {len(html)} | chars texto: {len(texto_visivel)}")

    assert not encontrados, f"BLOQUEIO DETECTADO em {nome}: {encontrados} -- pare e reporte, não contorne."
    print("  -> nenhum sinal de bloqueio.")

print("\nNenhum bloqueio detectado nas duas URLs -- prosseguindo.")

## Teste 1 — extrair a listagem estruturada

In [0]:
def extrair_listagem(html: str) -> list[dict]:
    soup = BeautifulSoup(html, "lxml")
    itens = []

    for card in soup.select("div.frw-column"):
        tag_titulo = card.select_one("h3.frw-heading--5 a")
        if not tag_titulo:
            continue

        titulo = tag_titulo.get_text(strip=True)
        url = urllib.parse.urljoin(BASE_URL, tag_titulo["href"])

        tipo, data_publicacao = None, None
        tag_tag = card.select_one(".frw-heading--tag")
        if tag_tag:
            texto_tag = tag_tag.get_text(" ", strip=True)
            if "/" in texto_tag:
                tipo, data_texto = [t.strip() for t in texto_tag.split("/", 1)]
                m = PADRAO_DATA_FITCH.search(data_texto)
                if m:
                    dia, mes_abrev, ano = m.groups()
                    mes = MESES_EN.get(mes_abrev.lower())
                    if mes:
                        data_publicacao = f"{ano}-{mes}-{dia.zfill(2)}"

        tag_resumo = card.select_one("p")
        resumo = tag_resumo.get_text(" ", strip=True) if tag_resumo else None

        itens.append({
            "titulo": titulo,
            "url": url,
            "tipo_publicacao": tipo,
            "published_at": data_publicacao,
            "resumo": resumo,
        })

    return itens


itens_racs = extrair_listagem(paginas_html["racs"])
itens_research = extrair_listagem(paginas_html["research"])

print(f"RACs: {len(itens_racs)} itens")
print(f"Research: {len(itens_research)} itens\n")

for nome, itens in [("RACS", itens_racs), ("RESEARCH", itens_research)]:
    print(f"--- {nome} (amostra) ---")
    for item in itens[:8]:
        print(f"[{item['published_at']}] ({item['tipo_publicacao']}) {item['titulo'][:75]}")
    print()

## Teste 2 — abrir uma notícia de cada tipo e checar paywall

**Resultado, confirmado com o texto completo lido (não assumido):**

- **RAC (Rating Action Commentary)**: totalmente aberto. A página de
  detalhe testada (Opea/Rede D'Or) trouxe **16.841 caracteres** de texto
  visível -- tabela de ações de rating, "PRINCIPAIS FUNDAMENTOS DO
  RATING" com a análise completa. Nenhum indício de paywall.
- **Research/Insight (ex.: Rating Report)**: página de detalhe é uma
  espécie de landing page curta (**2.412 caracteres** no exemplo testado,
  Banco Citibank) -- resumo de 1-2 frases, metadados (entidade, região,
  conteúdos relacionados), e um botão **"Access Report"**
  (`<a id="btn-1" class="frw-button">`, **sem `href`** -- ação disparada
  por JavaScript, não um link direto). Isso indica que o relatório
  completo fica atrás de autenticação/assinatura -- **não cliquei no
  botão nem investiguei mais a fundo**, por não ser o objetivo desta
  Fase 1 e para não me aproximar de nenhum fluxo de login.

Ou seja: **os dois casos coexistem na mesma fonte**, dependendo do tipo
de publicação -- RACs abertos, Research/Insights com conteúdo represado.

In [0]:
def extrair_texto_pagina(url: str) -> dict:
    driver.get(url)
    time.sleep(6)

    texto = driver.find_element(By.TAG_NAME, "body").text
    tem_botao_acesso = bool(driver.find_elements(
        By.XPATH, '//*[contains(text(), "Access Report") or contains(text(), "ACCESS REPORT")]'
    ))

    return {
        "url": url,
        "tamanho_texto": len(texto),
        "tem_botao_access_report": tem_botao_acesso,
        "amostra": texto[:1500],
    }


# Exemplo de RAC (aberto)
url_rac_exemplo = itens_racs[0]["url"]
detalhe_rac = extrair_texto_pagina(url_rac_exemplo)
print("=== RAC ===")
print(f"URL: {detalhe_rac['url']}")
print(f"Tamanho texto: {detalhe_rac['tamanho_texto']} chars")
print(f"Tem botão 'Access Report': {detalhe_rac['tem_botao_access_report']}")
print(detalhe_rac["amostra"][:800])

# Exemplo de Research (represado)
url_research_exemplo = itens_research[0]["url"]
detalhe_research = extrair_texto_pagina(url_research_exemplo)
print("\n=== RESEARCH ===")
print(f"URL: {detalhe_research['url']}")
print(f"Tamanho texto: {detalhe_research['tamanho_texto']} chars")
print(f"Tem botão 'Access Report': {detalhe_research['tem_botao_access_report']}")
print(detalhe_research["amostra"][:800])

driver.quit()

## Teste 3 — menções a empresas da lista canônica (`canonical_entidades.json`)

Cruza título + resumo de cada item da listagem contra os `aliases` de
`scripts/canonical_entidades.json`, via substring simples (não é NER,
não é matching de palavra inteira).

**Resultado da amostra de 48 itens (24 RACs + 24 Research)**: 3
correspondências brutas, 2 confirmadas reais lendo o resumo completo
(não só o título truncado):

- **RAC**: *"Fitch Afirma Rating 'AA(bra)' de Debêntures da Ecovias
  Ponte"* -- resumo completo confirma "Concessionária Ponte Rio-Niterói
  S.A. – Ecoponte (Ecovias Ponte)" -- match real com o canônico
  `CONCESSIONÁRIA PONTE RIO-NITERÓI S.A.` (alias `Ecoponte`).
- **Research**: *"Aumento de Capital da Aegea é Positivo, Mas Não Melhora
  Ratings"* -- resumo confirma "Aegea Saneamento e Participações S.A." --
  match real com `AEGEA SANEAMENTO E PARTICIPACOES S.A.`.
- **Falso positivo identificado**: *"Relatório de Pré-Distribuição - FIDC
  Consig Premium II"* bateu com `RUMO MALHA PAULISTA S/A` -- ao ler o
  resumo completo, nenhum termo relacionado à Rumo aparece; a lista de
  aliases da Rumo inclui palavras curtas e genéricas em português/inglês
  comuns (`Edge`, `Radar`, `Moove`, `Compass`, `Commit`) que colidem por
  coincidência de substring com texto de um relatório de fundo de
  recebíveis de crédito consignado sem relação nenhuma com a Rumo.

**Implicação para uma eventual Fase 2**: o valor real dessa fonte (achar
menções a empresas do portfólio dentro de um universo bem maior de
cobertura da Fitch) é genuíno e confirmado -- mas o matching por
substring simples usado aqui **não é confiável o suficiente para
produção sem revisão**. Aliases de 4+ caracteres que também são palavras
comuns (curtas ou não) precisam de matching por palavra inteira
(`\bpalavra\b`) no mínimo, e idealmente checagem contra a seção
`ENTITIES`/`RATING ACTIONS` da página de detalhe (estruturada, sem
ambiguidade) em vez de título+resumo em texto livre.

In [0]:
import json as _json

with open(
    "/Workspace/Shared/Research_Infra/Data-Ingestion-Pipeline-for-Kinea-Research-Infrastructure/"
    "scripts/canonical_entidades.json",
    encoding="utf-8",
) as f:
    canonicos = _json.load(f)

todos_aliases = []
for e in canonicos:
    for a in e["aliases"]:
        todos_aliases.append((a.lower(), e["nome_principal"]))


def acha_mencoes_ingenuo(texto: str) -> set:
    """Matching por substring simples -- ver ressalva acima sobre falsos
    positivos com aliases curtos/genéricos. Suficiente para sinalizar
    candidatos nesta Fase 1, não para produção sem revisão."""
    t = texto.lower()
    return {nome for alias, nome in todos_aliases if len(alias) >= 4 and alias in t}


total_com_match = 0
for nome_pagina, itens in [("RACS", itens_racs), ("RESEARCH", itens_research)]:
    print(f"--- {nome_pagina} ---")
    for item in itens:
        texto_busca = f"{item['titulo']} {item['resumo'] or ''}"
        achados = acha_mencoes_ingenuo(texto_busca)
        if achados:
            total_com_match += 1
            print(f"  [PORTFOLIO?: {sorted(achados)}] {item['titulo'][:80]}")

print(f"\nTotal de itens com correspondência bruta (antes de checar falso-positivo): {total_com_match} de {len(itens_racs) + len(itens_research)}")
print("Ver célula markdown acima: 2 de 3 confirmados reais lendo o resumo completo, 1 falso positivo identificado.")

## Conclusão da Fase 1

- **Sem bloqueio anti-bot** em nenhuma das duas URLs, com navegador real
  (Selenium + Chrome headless, cluster com init script resolvido).
- **Estrutura da listagem** confirmada e extraível: tipo, data, título,
  link, resumo (truncado) -- nome da empresa avaliada não é um campo
  separado, precisa inferir do título ou abrir o detalhe.
- **Paywall confirmado nos dois sentidos, não assumido**: RAC totalmente
  aberto (texto completo, tabela de ratings, fundamentos); Research/
  Insight represado atrás de um botão "Access Report" sem link direto
  (autenticação necessária) -- landing page com resumo curto acessível,
  conteúdo completo não.
- **Valor real da fonte confirmado**: 2 menções genuínas a empresas do
  portfólio canônico numa amostra de 48 itens (Ecoponte, Aegea) --
  mas o método de matching usado (substring simples) também produziu 1
  falso positivo por causa de aliases curtos/genéricos -- precisa de
  matching mais robusto numa eventual Fase 2.

**Nada foi registrado em `controle_fontes`** -- por instrução explícita,
esta etapa é só teste isolado. Aguardando decisão sobre integração antes
de prosseguir para Fase 2/3.